# 🔗 Prompt Chaining으로 더 똑똑한 답변 받기

이번 실습에서는 **Prompt Chaining** 패턴을 활용하여 2개 이상의 LLM을 순차적으로 연결하여 더 정교한 답변을 생성하는 워크플로우를 구현해봅니다.

> 📢 **Prompt Chaining이란?**
> 
> 여러 LLM 호출을 순차적으로 연결하여 각 단계의 출력이 다음 단계의 입력이 되는 패턴입니다.
> - **분해**: 복잡한 작업을 여러 단계로 나눔
> - **정제**: 각 단계에서 결과를 점진적으로 개선
> - **품질 향상**: 단일 호출보다 더 높은 품질의 결과 도출

### 🎯 학습 목표
1. LangGraph로 Sequential Workflow 구현하기
2. 노드 간 데이터 전달 방법 이해하기
3. 단계별 LLM 호출로 결과 품질 향상시키기

## 📋 목차

1. [환경 설정](#1-환경-설정)
2. [상태 정의](#2-상태-정의)
3. [체이닝 노드 구현](#3-체이닝-노드-구현)
4. [그래프 구성](#4-그래프-구성)
5. [실행 및 테스트](#5-실행-및-테스트)

---
## 1. 환경 설정

In [ ]:
# 필요한 패키지 설치
%pip install -qU langchain langgraph langchain-openai python-dotenv

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()

if not os.getenv("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = "your-openai-api-key"

print("✅ 환경 설정 완료!")

---
## 2. 상태 정의

Prompt Chaining에서는 각 단계의 결과를 저장할 상태가 필요합니다.

In [ ]:
from typing import Annotated, TypedDict
import operator

class ChainState(TypedDict):
    """Prompt Chaining을 위한 상태 정의"""
    user_input: str           # 사용자 입력
    step1_output: str         # 1단계: 초안 작성
    step2_output: str         # 2단계: 구조화
    step3_output: str         # 3단계: 최종 정제
    final_output: str         # 최종 결과

print("✅ 상태 정의 완료!")
print("   워크플로우: 초안 작성 → 구조화 → 최종 정제")

---
## 3. 체이닝 노드 구현

각 단계별로 다른 역할을 수행하는 노드를 구현합니다:
1. **draft_writer**: 초안 작성
2. **structurer**: 내용 구조화
3. **refiner**: 최종 정제

In [ ]:
from langchain_openai import ChatOpenAI
from langchain_core.messages import SystemMessage, HumanMessage

# LLM 초기화
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.7)

def draft_writer(state: ChainState) -> dict:
    """1단계: 초안 작성"""
    print("\n📝 Step 1: 초안 작성 중...")
    
    prompt = f"""다음 주제에 대해 자유롭게 아이디어를 브레인스토밍하고 초안을 작성해주세요.
형식이나 구조는 신경쓰지 말고, 모든 관련 내용을 포함해주세요.

주제: {state['user_input']}

초안:"""
    
    response = llm.invoke([HumanMessage(content=prompt)])
    return {"step1_output": response.content}

def structurer(state: ChainState) -> dict:
    """2단계: 내용 구조화"""
    print("\n🏗️ Step 2: 구조화 중...")
    
    prompt = f"""다음 초안을 잘 구조화된 형식으로 재구성해주세요.
- 명확한 섹션 구분
- 논리적 순서 배열
- 핵심 포인트 강조

초안:
{state['step1_output']}

구조화된 내용:"""
    
    response = llm.invoke([HumanMessage(content=prompt)])
    return {"step2_output": response.content}

def refiner(state: ChainState) -> dict:
    """3단계: 최종 정제"""
    print("\n✨ Step 3: 최종 정제 중...")
    
    prompt = f"""다음 내용을 최종적으로 다듬어주세요.
- 문장 표현 개선
- 불필요한 중복 제거
- 전문적이면서도 읽기 쉬운 톤 유지
- 한국어로 작성

구조화된 내용:
{state['step2_output']}

최종 결과:"""
    
    response = llm.invoke([HumanMessage(content=prompt)])
    return {"step3_output": response.content, "final_output": response.content}

print("✅ 체이닝 노드 구현 완료!")

---
## 4. 그래프 구성

Sequential 워크플로우:
```
START → draft_writer → structurer → refiner → END
```

In [ ]:
from langgraph.graph import StateGraph, START, END

# 그래프 생성
workflow = StateGraph(ChainState)

# 노드 추가
workflow.add_node("draft_writer", draft_writer)
workflow.add_node("structurer", structurer)
workflow.add_node("refiner", refiner)

# 순차적 엣지 연결
workflow.add_edge(START, "draft_writer")
workflow.add_edge("draft_writer", "structurer")
workflow.add_edge("structurer", "refiner")
workflow.add_edge("refiner", END)

# 컴파일
chain_agent = workflow.compile()

print("✅ Prompt Chaining 워크플로우 컴파일 완료!")

In [ ]:
# 그래프 시각화
from IPython.display import Image, display

try:
    display(Image(chain_agent.get_graph().draw_mermaid_png()))
except Exception as e:
    print(f"시각화 오류: {e}")
    print("\n그래프 구조: START → draft_writer → structurer → refiner → END")

---
## 5. 실행 및 테스트

In [ ]:
# 테스트: 블로그 포스트 작성
user_request = "AI 에이전트가 미래의 업무 방식을 어떻게 변화시킬지에 대한 블로그 포스트"

print(f"📄 요청: {user_request}")
print("=" * 60)

result = chain_agent.invoke({"user_input": user_request})

print("\n" + "=" * 60)
print("\n🎯 최종 결과:")
print("=" * 60)
print(result["final_output"])

In [ ]:
# 각 단계별 결과 확인
print("📋 단계별 결과 비교")
print("\n" + "=" * 60)
print("[Step 1: 초안]")
print(result["step1_output"][:500] + "...")
print("\n" + "=" * 60)
print("[Step 2: 구조화]")
print(result["step2_output"][:500] + "...")
print("\n" + "=" * 60)
print("[Step 3: 최종 정제]")
print(result["step3_output"][:500] + "...")

---
## 💡 정리

### Prompt Chaining의 장점

| 장점 | 설명 |
|------|------|
| **품질 향상** | 단계별 정제로 더 나은 결과물 |
| **복잡성 분해** | 어려운 작업을 단순한 단계로 나눔 |
| **디버깅 용이** | 각 단계의 결과를 확인하고 개선 가능 |
| **유연성** | 특정 단계만 수정하거나 교체 가능 |

### 활용 사례
- 📝 콘텐츠 작성 (브레인스토밍 → 초안 → 편집 → 최종)
- 🔍 분석 리포트 (데이터 수집 → 분석 → 인사이트 도출 → 요약)
- 💻 코드 생성 (요구사항 분석 → 설계 → 구현 → 리팩토링)

### 다음 단계
👉 **CH02._03.** MoA(Mixture of Agents)로 AI 답변 종합하기